In [1]:
import random

def generate_medical_history():
    conditions = ["Kidney disease", "Diabetes", "Hypertension", None]
    allergies = ["Penicillin", "Latex", None]
    medications = ["Blood thinners", "Insulin", None]

    return {
        "condition": random.choice(conditions),
        "allergy": random.choice(allergies),
        "medication": random.choice(medications)
    }

In [2]:
def emergency_risk_agent(state):
    history = generate_medical_history()

    prompt = f"""
    You are an emergency medical AI agent.

    Patient details:
    Admission type: {state['admission_type']}
    Department: {state['department']}
    Complications: {state['complications']}
    Length of stay: {state['stay']} days

    Medical history:
    Chronic condition: {history['condition']}
    Allergy: {history['allergy']}
    Medication: {history['medication']}

    In an emergency situation:
    1. What treatment risks should doctors be aware of?
    2. What medications or procedures should be avoided?
    3. What immediate precautions are necessary?

    Keep response concise and clinical.
    """

    return nova_call(prompt)  # reuse your nova function

In [3]:
def run_pipeline_with_ai(row):
    state = interpret_patient(row)
    issues = detect_bottlenecks(state)
    actions = recommend_actions(issues)

    coordination_explanation = nova_explain(
        state["patient_id"], issues, actions
    )

    emergency_alert = emergency_risk_agent(state)

    return {
        "patient_id": state["patient_id"],
        "issues": issues,
        "actions": actions,
        "coordination_explanation": coordination_explanation,
        "emergency_risk_alert": emergency_alert
    }

In [4]:
import pandas as pd
import random
import json
import boto3

In [5]:
df = pd.read_csv("healthcare_patient_journey.csv")
df.head()

,patient_id,age,gender,chronic_condition,admission_type,department,wait_time_min,length_of_stay_days,procedures_count,medication_count,complications,discharge_status,readmitted_30d,total_cost_€,satisfaction_score
0,1,69,male,0,scheduled,Neurology,41,2,0,3,1,referred,1,1440,2
1,2,38,male,0,emergency,Oncology,17,3,1,2,0,recovered,0,2060,3
2,3,81,male,0,scheduled,Neurology,40,2,3,2,0,recovered,0,2110,3
3,4,67,female,1,emergency,ER,7,4,5,9,0,recovered,0,4070,3
4,5,88,male,1,emergency,Cardiology,34,3,7,5,0,recovered,1,3800,3


In [6]:
def interpret_patient(row):
    return {
        "patient_id": row["patient_id"],
        "admission_type": row["admission_type"],
        "department": row["department"],
        "wait_time": row["wait_time_min"],
        "stay": row["length_of_stay_days"],
        "procedures": row["procedures_count"],
        "complications": row["complications"],
        "readmission": row["readmitted_30d"],
        "satisfaction": row["satisfaction_score"]
    }

In [7]:
def detect_bottlenecks(state):
    issues = []

    if state["wait_time"] > 30:
        issues.append("Admission delay")

    if state["stay"] > 5:
        issues.append("Extended hospitalization")

    if state["complications"] == 1:
        issues.append("Clinical complication")

    if state["readmission"] == 1:
        issues.append("Poor discharge coordination")

    return issues

In [8]:
def recommend_actions(issues):
    actions = []

    mapping = {
        "Admission delay": "Optimize triage scheduling",
        "Extended hospitalization": "Initiate discharge planning",
        "Clinical complication": "Escalate specialist review",
        "Poor discharge coordination": "Schedule post-discharge follow-up"
    }

    for issue in issues:
        if issue in mapping:
            actions.append(mapping[issue])

    return actions

In [9]:
def generate_medical_history():
    return {
        "condition": random.choice(["Kidney disease", "Diabetes", "Hypertension", None]),
        "allergy": random.choice(["Penicillin", "Latex", None]),
        "medication": random.choice(["Blood thinners", "Insulin", None])
    }

In [16]:
def severity_score(state, issues):

    score = 0

    if state["wait_time"] > 30:
        score += 1

    if state["stay"] > 5:
        score += 1

    if state["complications"] == 1:
        score += 2

    if state["readmission"] == 1:
        score += 2

    if score >= 4:
        return "CRITICAL"
    elif score >= 2:
        return "HIGH"
    elif score == 1:
        return "MEDIUM"
    else:
        return "LOW"

In [10]:
bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

import boto3
import json

bedrock = boto3.client("bedrock-runtime", region_name="us-east-1")

def call_nova(prompt):

    body = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"text": prompt}
                ]
            }
        ],
        "inferenceConfig": {
            "maxTokens": 400,
            "temperature": 0.3
        }
    }

    response = bedrock.invoke_model(
        modelId="amazon.nova-lite-v1:0",
        body=json.dumps(body),
        contentType="application/json"
    )

    result = json.loads(response["body"].read())

    return result["output"]["message"]["content"][0]["text"]

In [ ]:
def coordination_agent(state, issues, actions,severity):

    if not issues:
        return "No coordination issues detected."

    prompt = f"""
    You are a hospital care coordination AI.

    Patient ID: {state['patient_id']}
    Issues: {issues}
    Recommended Actions: {actions}

    Explain bottlenecks and why actions are necessary.
    Keep concise and clinical.

    Severity: {severity}
    """

    return call_nova(prompt)

In [12]:
def emergency_agent(state):

    history = generate_medical_history()

    prompt = f"""
    You are an emergency medical AI agent.

    Patient details:
    Admission type: {state['admission_type']}
    Department: {state['department']}
    Complications: {state['complications']}
    Length of stay: {state['stay']} days

    Medical history:
    Chronic condition: {history['condition']}
    Allergy: {history['allergy']}
    Medication: {history['medication']}

    Identify emergency risks and precautions.
    """

    return call_nova(prompt)

In [17]:
def run_patient_pipeline(row):

    state = interpret_patient(row)
    issues = detect_bottlenecks(state)
    actions = recommend_actions(issues)
    severity = severity_score(state,issues)

    coordination = coordination_agent(state, issues, actions)
    emergency = emergency_agent(state)

    return {
        "patient_id": state["patient_id"],
        "issues": issues,
        "actions": actions,
        "severity": severity,
        "coordination_explanation": coordination,
        "emergency_alert": emergency
    }

In [18]:
run_patient_pipeline(df.iloc[0])

{'patient_id': np.int64(1),
 'issues': ['Admission delay',
  'Clinical complication',
  'Poor discharge coordination'],
 'actions': ['Optimize triage scheduling',
  'Escalate specialist review',
  'Schedule post-discharge follow-up'],
 'severity': 'CRITICAL',
 'coordination_explanation': '**Bottlenecks:**\n\n1. **Admission Delay:** Prolonged wait times at the triage stage can lead to extended patient discomfort, increased risk of clinical deterioration, and overall dissatisfaction.\n   \n2. **Clinical Complication:** Delays in admission and suboptimal initial care can result in the exacerbation of existing conditions or the development of new complications, complicating the treatment process.\n\n3. **Poor Discharge Coordination:** Ineffective discharge planning can result in fragmented care, increased readmission rates, and poor patient outcomes post-discharge.\n\n**Recommended Actions and Necessity:**\n\n1. **Optimize Triage Scheduling:** Streamlining the triage process ensures timely

In [19]:
sample = df.head(5)
results = sample.apply(run_patient_pipeline, axis=1)
results = pd.DataFrame(results.tolist())
results

,patient_id,issues,actions,severity,coordination_explanation,emergency_alert
0,1,"[Admission delay, Clinical complication, Poor ...","[Optimize triage scheduling, Escalate speciali...",CRITICAL,**Bottlenecks:**\n\n1. **Admission Delay:** Pr...,"Certainly, here are the identified emergency r..."
1,2,[],[],LOW,No coordination issues detected.,Certainly. Here are the identified emergency r...
2,3,[Admission delay],[Optimize triage scheduling],MEDIUM,**Patient ID: 3**\n\n**Issues:**\n- Admission ...,Given the patient's medical history and the co...
3,4,[],[],LOW,No coordination issues detected.,"Given the patient's details, here are the emer..."
4,5,"[Admission delay, Poor discharge coordination]","[Optimize triage scheduling, Schedule post-dis...",HIGH,**Bottlenecks:**\n\n1. **Admission Delay:**\n ...,Certainly. Here are the identified emergency r...


In [20]:
results.to_csv("results.csv", index=False)